# Sentence-Transformers

Imagine a robot that turns each sentence into a magic number vector.
If two vectors point in a similar direction, the sentences have similar meaning.

We will use **Sentence-Transformers** because it is made for sentence similarity.

Core dependencies (install explicitly)

- sentence-transformers
- torch (CPU or CUDA build matching your runtime)
- transformers
- huggingface-hub
- tokenizers
- safetensors
- numpy
- scikit-learn
- scipy
- tqdm
- Pillow (used by some transformer stacks)

Offline model requirement (critical)

- Pre-download model repo (for example BAAI/bge-base-en-v1.5) outside Palantir.
- Copy full model folder into your internal artifact store.
- Load by local path, not model ID: SentenceTransformer("/path/to/local/model")

Offline install pattern

- Internet machine: pip download -d wheelhouse sentence-transformers torch transformers scikit-learn scipy numpy
- Transfer wheelhouse/ + model folder into Palantir.
- Offline install: pip install --no-index --find-links=wheelhouse sentence-transformers torch transformers scikit-learn scipy numpy
- Set offline env vars: HF_HUB_OFFLINE=1 and TRANSFORMERS_OFFLINE=1

In [29]:
from sentence_transformers import SentenceTransformer, util
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

## 1) Load a sentence model

`all-MiniLM-L6-v2` is small, fast, and great for learning.

In [30]:
model = SentenceTransformer("all-MiniLM-L6-v2")
print("Model loaded")

Model loaded


## 2) Convert sentences to vectors

In [31]:
sentences = [
    "I love this movie.",
    "This film is fantastic.",
    "The weather is very hot today.",
    "It's sunny and warm.",
    "That movie was terrible."
]

embeddings = model.encode(sentences, convert_to_tensor=True)
print("Embeddings shape:", embeddings.shape)

Embeddings shape: torch.Size([5, 384])


## 3) Cosine similarity using sentence-transformers util

In [32]:
sim_matrix = util.cos_sim(embeddings, embeddings)
print(sim_matrix)

tensor([[ 1.0000,  0.7148,  0.0360,  0.1755,  0.5155],
        [ 0.7148,  1.0000,  0.0981,  0.2101,  0.5386],
        [ 0.0360,  0.0981,  1.0000,  0.6954, -0.0553],
        [ 0.1755,  0.2101,  0.6954,  1.0000,  0.0258],
        [ 0.5155,  0.5386, -0.0553,  0.0258,  1.0000]])


## 4) Cosine similarity using sklearn

This gives the same idea, with another library.

In [33]:
emb_np = embeddings.cpu().numpy()
sk_sim = cosine_similarity(emb_np)

print("sklearn similarity matrix:")
print(np.round(sk_sim, 4))

sklearn similarity matrix:
[[ 1.      0.7148  0.036   0.1755  0.5155]
 [ 0.7148  1.      0.0981  0.2101  0.5386]
 [ 0.036   0.0981  1.      0.6954 -0.0553]
 [ 0.1755  0.2101  0.6954  1.      0.0258]
 [ 0.5155  0.5386 -0.0553  0.0258  1.    ]]


## 5) Easy-to-read pair scores

In [34]:
pairs = [(0, 1), (0, 2), (1, 2)]
for i, j in pairs:
    print(f"{sentences[i]}  <->  {sentences[j]}")
    print("score:", round(float(sk_sim[i, j]), 4))
    print("-" * 60)

I love this movie.  <->  This film is fantastic.
score: 0.7148
------------------------------------------------------------
I love this movie.  <->  The weather is very hot today.
score: 0.036
------------------------------------------------------------
This film is fantastic.  <->  The weather is very hot today.
score: 0.0981
------------------------------------------------------------


---
---
---

## Quick recap

- Sentence-Transformers is better for sentence similarity than raw transformer hidden-state averaging.
- Higher cosine score means closer meaning.
- A similarity matrix helps compare many sentences at once.

---
---
---

# Step-by-step: Text format classification (privacy-safe)

This section uses only these source columns:
- `full_item` (raw text)
- `stage1_code` (stage-1 code)
- `stage2_code` (stage-2 code)

Preprocessing creates:
- `combine_text` (cleaned text for training)
- `combined_target` (single combined label)

You get **two implementations in the same notebook**:
1. Two-stage classifier: stage-1 and stage-2 trained separately
2. Single-stage classifier: combined target trained once

## Step 1: Import libraries and initialize NLP preprocessor

In [35]:
import re
import numpy as np
import pandas as pd
import spacy

from sentence_transformers import SentenceTransformer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

from sklearn.feature_extraction.text import HashingVectorizer
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.pipeline import Pipeline

nlp_pre = spacy.load("en_core_web_sm", disable=["parser", "ner"])
print("Preprocessor ready")

Preprocessor ready


## Step 2: Load data and create required training columns

This step:
- picks `full_item`, `stage1_code`, `stage2_code`
- preprocesses text into `combine_text`
- builds `combined_target`
- creates privacy-safe labels (`category_stage1`, `category_stage2`)

In [36]:
import re
import spacy

df_raw = pd.read_csv("../data/text_format_raw_training_data.csv")

if "nlp_pre" not in globals():
    nlp_pre = spacy.load("en_core_web_sm", disable=["parser", "ner"])

df = df_raw.copy()
required_cols = ["full_item", "stage1_code", "stage2_code"]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

df = df[required_cols].dropna().drop_duplicates().reset_index(drop=True)

def normalize_2digit(value: str, pick: str = "first") -> str:
    parts = re.findall(r"\d{2}", str(value))
    if not parts:
        digits = re.sub(r"\D", "", str(value))
        if len(digits) >= 2:
            return digits[:2]
        return digits.zfill(2)
    if pick == "second":
        return parts[1] if len(parts) > 1 else parts[0]
    return parts[0]

def preprocess_text(text: str) -> str:
    text = str(text).lower()
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    doc = nlp_pre(text)
    tokens = [
        tok.lemma_.strip()
        for tok in doc
        if tok.is_alpha and not tok.is_stop and tok.lemma_.strip()
    ]
    return " ".join(tokens)

df["stage1_code"] = df["stage1_code"].astype(str).apply(lambda x: normalize_2digit(x, pick="first"))
df["stage2_code"] = df["stage2_code"].astype(str).apply(lambda x: normalize_2digit(x, pick="second"))
df["combined_target"] = df["stage1_code"] + " " + df["stage2_code"]

df["combine_text"] = df["full_item"].astype(str).apply(preprocess_text)
df = df[df["combine_text"].str.len() > 0].reset_index(drop=True)

# privacy-safe category labels for display/training outputs
stage1_unique = sorted(df["stage1_code"].unique())
stage2_unique = sorted(df["stage2_code"].unique())
stage1_map = {code: f"category_stage1_{i:02d}" for i, code in enumerate(stage1_unique, start=1)}
stage2_map = {code: f"category_stage2_{i:03d}" for i, code in enumerate(stage2_unique, start=1)}

df["category_stage1"] = df["stage1_code"].map(stage1_map)
df["category_stage2"] = df["stage2_code"].map(stage2_map)
df["category_combined"] = df["category_stage1"] + "__" + df["category_stage2"]

category_stage1_to_code = {v: k for k, v in stage1_map.items()}
category_stage2_to_code = {v: k for k, v in stage2_map.items()}

output_path = "../data/text_format_training_data_preprocessed.csv"
df.to_csv(output_path, index=False)

print("Rows after cleanup:", len(df))
print("Saved transformed training data to:", output_path)
print(df[["full_item", "combine_text", "stage1_code", "stage2_code", "combined_target"]].head(3))

Rows after cleanup: 2389
Saved transformed training data to: ../data/text_format_training_data_preprocessed.csv
                                           full_item  \
0  Subcontractor bid for condensing high-efficien...   
1  SCOPE OF WORK DESCRIPTION: Supply and install ...   
2  VENDOR SUBMITTAL: Material submittal for veget...   

                                        combine_text stage1_code stage2_code  \
0  subcontractor bid condense high efficiency boi...          23          70   
1  scope work description supply install evaporat...          23          60   
2  vendor submittal material submittal vegetate g...          07          50   

  combined_target  
0           23 70  
1           23 60  
2           07 50  


## Step 3: Prepare splits for both implementations

In [37]:
X = df["combine_text"].tolist()
y_stage1 = df["category_stage1"].tolist()
y_stage2 = df["category_stage2"].tolist()
y_combined = df["category_combined"].tolist()

(
    X_train,
    X_test,
    y_stage1_train,
    y_stage1_test,
    y_stage2_train,
    y_stage2_test,
    y_combined_train,
    y_combined_test,
) = train_test_split(
    X,
    y_stage1,
    y_stage2,
    y_combined,
    test_size=0.2,
    random_state=42,
    stratify=y_combined,
)

print("Train size:", len(X_train))
print("Test size:", len(X_test))

Train size: 1911
Test size: 478


## Step 4: Create sentence embeddings from `combine_text`

For offline environments, set `MODEL_NAME_OR_PATH` to your local model directory.

In [38]:
MODEL_NAME_OR_PATH = "sentence-transformers/all-MiniLM-L6-v2"  # replace with local path in restricted env
embedder = SentenceTransformer(MODEL_NAME_OR_PATH)

X_train_emb = embedder.encode(X_train, show_progress_bar=True)
X_test_emb = embedder.encode(X_test, show_progress_bar=True)

print("Train embedding shape:", X_train_emb.shape)
print("Test embedding shape:", X_test_emb.shape)

Batches: 100%|██████████| 15/15 [00:05<00:00,  2.80it/s]

Train embedding shape: (1911, 384)
Test embedding shape: (478, 384)


## Implementation 1: Two-stage model (stage-1 and stage-2 trained separately)

This is useful when you want independent control over each stage.

In [39]:
model_stage1 = LogisticRegression(max_iter=3000)
model_stage2 = LogisticRegression(max_iter=3000)

model_stage1.fit(X_train_emb, y_stage1_train)
model_stage2.fit(X_train_emb, y_stage2_train)

pred_stage1 = model_stage1.predict(X_test_emb)
pred_stage2 = model_stage2.predict(X_test_emb)

print("Two-stage stage-1 accuracy:", round(accuracy_score(y_stage1_test, pred_stage1), 4))
print("Two-stage stage-2 accuracy:", round(accuracy_score(y_stage2_test, pred_stage2), 4))

print("\nStage-1 report")
print(classification_report(y_stage1_test, pred_stage1, zero_division=0))

print("\nStage-2 report")
print(classification_report(y_stage2_test, pred_stage2, zero_division=0))

Two-stage stage-1 accuracy: 0.9854
Two-stage stage-2 accuracy: 0.9561

Stage-1 report
                    precision    recall  f1-score   support

category_stage1_01       1.00      0.97      0.98        60
category_stage1_02       0.95      1.00      0.98        60
category_stage1_03       0.98      1.00      0.99        60
category_stage1_04       0.97      1.00      0.98        59
category_stage1_05       1.00      0.95      0.97        60
category_stage1_06       0.98      1.00      0.99        60
category_stage1_07       1.00      0.97      0.98        60
category_stage1_08       1.00      1.00      1.00        59

          accuracy                           0.99       478
         macro avg       0.99      0.99      0.99       478
      weighted avg       0.99      0.99      0.99       478


Stage-2 report
                     precision    recall  f1-score   support

category_stage2_001       1.00      1.00      1.00        20
category_stage2_002       0.94      0.95      0.94  

## Implementation 2: Single-stage model (combined target)

This model predicts one combined class (`category_stage1__category_stage2`) in one shot.

In [40]:
model_single = LogisticRegression(max_iter=3000)
model_single.fit(X_train_emb, y_combined_train)

pred_combined = model_single.predict(X_test_emb)
print("Single-stage combined accuracy:", round(accuracy_score(y_combined_test, pred_combined), 4))
print(classification_report(y_combined_test, pred_combined, zero_division=0))

# split combined prediction back into stage-1 and stage-2
pred_single_stage1 = [x.split("__")[0] for x in pred_combined]
pred_single_stage2 = [x.split("__")[1] for x in pred_combined]

print("\nSingle-stage decoded stage-1 accuracy:", round(accuracy_score(y_stage1_test, pred_single_stage1), 4))
print("Single-stage decoded stage-2 accuracy:", round(accuracy_score(y_stage2_test, pred_single_stage2), 4))

Single-stage combined accuracy: 0.9854
                                         precision    recall  f1-score   support

category_stage1_01__category_stage2_002       1.00      1.00      1.00        20
category_stage1_01__category_stage2_003       1.00      0.95      0.97        20
category_stage1_01__category_stage2_004       1.00      1.00      1.00        20
category_stage1_02__category_stage2_002       0.87      1.00      0.93        20
category_stage1_02__category_stage2_004       1.00      0.95      0.97        20
category_stage1_02__category_stage2_006       1.00      0.95      0.97        20
category_stage1_03__category_stage2_002       1.00      1.00      1.00        20
category_stage1_03__category_stage2_003       1.00      1.00      1.00        20
category_stage1_03__category_stage2_006       1.00      1.00      1.00        20
category_stage1_04__category_stage2_002       1.00      1.00      1.00        19
category_stage1_04__category_stage2_006       1.00      1.00      1.0

## Recommended production approach: confidence-based hybrid refinement

You can keep your current ML pipeline and only use embeddings for low-confidence cases:
1. Base model: `HashingVectorizer + LinearSVC + CalibratedClassifierCV`
2. If confidence < threshold, route to embedding classifier

This gives speed + stability for easy samples, and better recovery on hard samples.

In [41]:
# Base pipeline similar to your current production setup
base_pipe = Pipeline(
    steps=[
        ("vect", HashingVectorizer(n_features=2**18, alternate_sign=False, norm="l2")),
        (
            "clf",
            CalibratedClassifierCV(
                estimator=LinearSVC(),
                method="sigmoid",
                cv=3,
            ),
        ),
    ]
)

base_pipe.fit(X_train, y_combined_train)
base_proba = base_pipe.predict_proba(X_test)
base_pred = base_pipe.classes_[np.argmax(base_proba, axis=1)]
base_conf = np.max(base_proba, axis=1)

# Embedding fallback model (single-stage combined target)
emb_refiner = LogisticRegression(max_iter=3000)
emb_refiner.fit(X_train_emb, y_combined_train)
emb_pred = emb_refiner.predict(X_test_emb)

CONF_THRESHOLD = 0.80  # configurable
final_pred = np.where(base_conf >= CONF_THRESHOLD, base_pred, emb_pred)

print("Base-only combined accuracy:", round(accuracy_score(y_combined_test, base_pred), 4))
print("Hybrid combined accuracy:", round(accuracy_score(y_combined_test, final_pred), 4))
print("Routed to embedding (%):", round(float((base_conf < CONF_THRESHOLD).mean() * 100), 2))

# Optional: helper for single-text inference with confidence gating
def predict_hybrid(text: str, threshold: float = 0.80):
    t = preprocess_text(text)
    base_p = base_pipe.predict_proba([t])[0]
    idx = int(np.argmax(base_p))
    base_label = base_pipe.classes_[idx]
    conf = float(base_p[idx])

    if conf >= threshold:
        final_label = base_label
        source = "base"
    else:
        emb = embedder.encode([t])
        final_label = emb_refiner.predict(emb)[0]
        source = "embedding_refiner"

    stage1_cat, stage2_cat = final_label.split("__")

    stage1_code = category_stage1_to_code.get(stage1_cat)
    if stage1_code is None:
        m = df.loc[df["category_stage1"] == stage1_cat, "stage1_code"]
        stage1_code = str(m.iloc[0]) if len(m) else "00"

    stage2_code = category_stage2_to_code.get(stage2_cat)
    if stage2_code is None:
        m = df.loc[df["category_stage2"] == stage2_cat, "stage2_code"]
        stage2_code = str(m.iloc[0]) if len(m) else "00"

    stage1_code = str(stage1_code).zfill(2)
    stage2_code = str(stage2_code).zfill(2)
    combined_target = f"{stage1_code} {stage2_code}"

    return {
        "final_label": final_label,
        "pred_stage1": stage1_code,
        "pred_stage2": stage2_code,
        "pred_combined_target": combined_target,
        "base_confidence": round(conf, 4),
        "prediction_source": source,
    }

example = "supply and install air distribution ductwork with balancing"
print(predict_hybrid(example, threshold=CONF_THRESHOLD))

Base-only combined accuracy: 0.9979
Hybrid combined accuracy: 0.9979
Routed to embedding (%): 1.46
{'final_label': np.str_('category_stage1_07__category_stage2_004'), 'pred_stage1': '23', 'pred_stage2': '30', 'pred_combined_target': '23 30', 'base_confidence': 0.918, 'prediction_source': 'base'}


## Benchmark: LogisticRegression vs LinearSVC+Calibrated (embeddings)

This benchmark compares two embedding classifiers on the same train/test split.

Metrics:
- Accuracy
- Macro F1
- Training time (seconds)
- Prediction time (seconds)

In [42]:
from time import perf_counter
from sklearn.metrics import accuracy_score, f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV

# Ensure these variables are available from previous steps:
# X_train_emb, X_test_emb, y_combined_train, y_combined_test

results = []

models = {
    "logistic_regression": LogisticRegression(max_iter=3000),
    "linear_svc_calibrated": CalibratedClassifierCV(
        estimator=LinearSVC(),
        method="sigmoid",
        cv=3,
    ),
}

for name, clf in models.items():
    t0 = perf_counter()
    clf.fit(X_train_emb, y_combined_train)
    train_time = perf_counter() - t0

    t1 = perf_counter()
    pred = clf.predict(X_test_emb)
    pred_time = perf_counter() - t1

    acc = accuracy_score(y_combined_test, pred)
    f1 = f1_score(y_combined_test, pred, average="macro", zero_division=0)

    results.append(
        {
            "model": name,
            "accuracy": round(float(acc), 4),
            "macro_f1": round(float(f1), 4),
            "train_time_sec": round(float(train_time), 4),
            "predict_time_sec": round(float(pred_time), 4),
        }
    )

benchmark_df = pd.DataFrame(results).sort_values(["macro_f1", "accuracy"], ascending=False)
print(benchmark_df.to_string(index=False))

                model  accuracy  macro_f1  train_time_sec  predict_time_sec
linear_svc_calibrated    1.0000    1.0000          5.0743            0.0283
  logistic_regression    0.9854    0.9855          1.9217            0.0020
